# PhiUSIIL Phishing URL Detection
## Step 3 — Data Preprocessing, Applied EDA & Feature Engineering

**Dataset:** `PhiUSIIL_Phishing_URL_Dataset.csv`

### Objectives
This notebook implements the capstone Step 3 workflow:

1. Load and validate the PhiUSIIL dataset.
2. Recode the target so **phishing = 1** and **legitimate = 0**.
3. Assess missing values, duplicates, invalid values, outliers, and leakage risks.
4. Normalize URL/domain strings for duplicate and group analysis.
5. Create **domain-grouped train/validation/test splits**.
6. Perform applied EDA focused on phishing detection.
7. Engineer safe **URL-only / pre-click** lexical features.
8. Engineer additional full-feature analytical variables.
9. Perform feature selection using correlation filtering, mutual information, and L1 Logistic Regression.
10. Apply PCA and optional t-SNE visualization.
11. Run an optional tree-based feature-importance / SHAP diagnostic.
12. Save processed outputs and metadata for reproducibility.

> **Safety:** This notebook does **not visit live URLs** or fetch webpage content. It uses only the information already contained in the CSV.

In [ ]:
from pathlib import Path
import json
import re
import hashlib
import ipaddress
import warnings
from urllib.parse import urlsplit, parse_qsl

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

RANDOM_STATE = 42
TARGET = "is_phishing"

print("Environment ready.")

## 1. Locate and load the dataset

In [ ]:
candidate_paths = [
    Path("PhiUSIIL_Phishing_URL_Dataset.csv"),
    Path("data/raw/PhiUSIIL_Phishing_URL_Dataset.csv"),
    Path("../data/raw/PhiUSIIL_Phishing_URL_Dataset.csv"),
    Path("/mnt/data/PhiUSIIL_Phishing_URL_Dataset.csv"),
]

DATA_PATH = next((p for p in candidate_paths if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "PhiUSIIL_Phishing_URL_Dataset.csv was not found. "
        "Place it beside this notebook or under data/raw/."
    )

OUTPUT_DIR = Path("outputs_step3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Loaded: {DATA_PATH.resolve()}")
print(f"Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]:,} columns")
display(df_raw.head())

In [ ]:
def sha256sum(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

print("SHA-256:", sha256sum(DATA_PATH))
print("Memory usage (MB):", round(df_raw.memory_usage(deep=True).sum() / 1024**2, 2))

## 2. Schema review and target recoding

In [ ]:
important_fields = [
    "FILENAME", "URL", "Domain", "TLD", "Title", "label",
    "URLLength", "DomainLength", "IsDomainIP", "TLDLength",
    "NoOfSubDomain", "IsHTTPS"
]

missing_important = [c for c in important_fields if c not in df_raw.columns]

if missing_important:
    print("Warning - expected fields not found:", missing_important)
else:
    print("All important schema fields are present.")

schema = pd.DataFrame({
    "column": df_raw.columns,
    "dtype": df_raw.dtypes.astype(str).values,
    "non_null": df_raw.notna().sum().values,
    "missing": df_raw.isna().sum().values,
    "unique": [df_raw[c].nunique(dropna=True) for c in df_raw.columns],
})

display(schema)

In [ ]:
# Original PhiUSIIL target:
# label = 1 -> legitimate
# label = 0 -> phishing
#
# Project target:
# is_phishing = 1 -> phishing
# is_phishing = 0 -> legitimate

if "label" not in df_raw.columns:
    raise KeyError("Expected target column 'label' was not found.")

label_values = set(pd.Series(df_raw["label"]).dropna().unique())
print("Original label values:", label_values)

if not label_values.issubset({0, 1}):
    raise ValueError("The 'label' column contains values outside {0, 1}.")

df = df_raw.copy()
df[TARGET] = 1 - df["label"].astype(int)

target_summary = (
    df[TARGET]
    .value_counts(dropna=False)
    .rename_axis("is_phishing")
    .to_frame("count")
)
target_summary["percent"] = 100 * target_summary["count"] / len(df)

display(target_summary)

## 3. Data-quality assessment

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (100 * missing / len(df)).round(4)

blank_counts = pd.Series(0, index=df.columns, dtype="int64")
for c in df.select_dtypes(include="object").columns:
    blank_counts.loc[c] = df[c].astype(str).str.strip().eq("").sum()

numeric_cols = df.select_dtypes(include=np.number).columns
inf_counts = pd.Series({
    c: np.isinf(pd.to_numeric(df[c], errors="coerce")).sum()
    for c in numeric_cols
})

quality_missing = pd.DataFrame({
    "missing_count": missing.reindex(df.columns),
    "missing_percent": missing_pct.reindex(df.columns),
    "blank_strings": blank_counts.reindex(df.columns).fillna(0).astype(int),
    "infinite_values": inf_counts.reindex(df.columns).fillna(0).astype(int),
}).sort_values(
    ["missing_count", "blank_strings", "infinite_values"],
    ascending=False
)

display(quality_missing.head(25))

In [ ]:
exact_duplicate_rows = df.duplicated().sum()
print(f"Exact duplicate rows: {exact_duplicate_rows:,}")

if "URL" in df.columns:
    url_counts = df["URL"].astype(str).value_counts()
    print(f"URL values appearing more than once: {(url_counts > 1).sum():,}")

if "Domain" in df.columns:
    domain_counts = df["Domain"].astype(str).value_counts()
    print(f"Domains appearing more than once: {(domain_counts > 1).sum():,}")

if {"URL", TARGET}.issubset(df.columns):
    conflicting_raw_urls = (
        df.groupby("URL")[TARGET]
        .nunique()
        .loc[lambda s: s > 1]
    )
    print(f"Raw URLs with conflicting labels: {len(conflicting_raw_urls):,}")

In [ ]:
candidate_binary = [
    "IsDomainIP", "HasObfuscation", "IsHTTPS", "HasTitle", "HasFavicon",
    "Robots", "IsResponsive", "HasDescription", "HasExternalFormSubmit",
    "HasSocialNet", "HasSubmitButton", "HasHiddenFields", "HasPasswordField",
    "Bank", "Pay", "Crypto", "HasCopyrightInfo", "label", TARGET
]
candidate_binary = [c for c in candidate_binary if c in df.columns]

binary_audit = []
for c in candidate_binary:
    vals = sorted(pd.Series(df[c]).dropna().unique().tolist())
    binary_audit.append({
        "variable": c,
        "unique_values": vals[:20],
        "n_unique": len(vals),
        "valid_0_1_only": set(vals).issubset({0, 1})
    })

display(pd.DataFrame(binary_audit))

In [ ]:
count_like_cols = [
    c for c in df.select_dtypes(include=np.number).columns
    if c.startswith("NoOf") or c in [
        "URLLength", "DomainLength", "TLDLength",
        "LineOfCode", "LargestLineLength"
    ]
]

negative_audit = pd.DataFrame({
    "variable": count_like_cols,
    "negative_count": [(df[c] < 0).sum() for c in count_like_cols]
}).sort_values("negative_count", ascending=False)

display(negative_audit.head(30))

## 4. URL/domain normalization and leakage checks

In [ ]:
def normalize_url(value):
    # Used only for duplicate/group analysis; raw URL is preserved.
    if pd.isna(value):
        return np.nan

    text = str(value).strip()
    if not text:
        return np.nan

    parse_text = text if "://" in text else "http://" + text

    try:
        p = urlsplit(parse_text)
        scheme = (p.scheme or "").lower()
        host = (p.hostname or "").lower().rstrip(".")
        port = f":{p.port}" if p.port else ""
        path = p.path or "/"
        query = f"?{p.query}" if p.query else ""

        if (scheme == "http" and port == ":80") or (scheme == "https" and port == ":443"):
            port = ""

        return f"{scheme}://{host}{port}{path}{query}"
    except Exception:
        return text.lower()


def normalize_domain(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip().lower()
    text = re.sub(r"^https?://", "", text)
    text = text.split("/")[0]
    text = text.split(":")[0]
    return text.rstrip(".")


df["URL_normalized"] = (
    df["URL"].map(normalize_url) if "URL" in df.columns else np.nan
)

df["Domain_normalized"] = (
    df["Domain"].map(normalize_domain)
    if "Domain" in df.columns
    else df["URL_normalized"].map(
        lambda x: urlsplit(x).hostname if pd.notna(x) else np.nan
    )
)

print("Unique normalized URLs:", df["URL_normalized"].nunique(dropna=True))
print("Unique normalized domains:", df["Domain_normalized"].nunique(dropna=True))

conflicting_norm_urls = (
    df.dropna(subset=["URL_normalized"])
    .groupby("URL_normalized")[TARGET]
    .nunique()
    .loc[lambda s: s > 1]
)

print("Normalized URLs with conflicting labels:", len(conflicting_norm_urls))

In [ ]:
leakage_review_features = [
    "URLSimilarityIndex", "CharContinuationRate",
    "TLDLegitimateProb", "URLCharProb",
    "DomainTitleMatchScore", "URLTitleMatchScore"
]
leakage_review_features = [c for c in leakage_review_features if c in df.columns]

if leakage_review_features:
    leakage_corr = (
        df[leakage_review_features + [TARGET]]
        .corr(numeric_only=True)[TARGET]
        .drop(TARGET)
        .sort_values(key=np.abs, ascending=False)
        .to_frame("correlation_with_target")
    )
    display(leakage_corr)

print(
    "Treat unusually predictive similarity/probability variables as leakage-risk "
    "candidates and test them in ablation experiments."
)

## 5. Cleaning decisions

In [ ]:
df_clean = df.copy()

before = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df_clean):,} exact duplicate rows.")

excluded_from_model = [c for c in ["FILENAME", "label"] if c in df_clean.columns]

print("Columns excluded from model inputs:", excluded_from_model)
print("Clean analysis shape:", df_clean.shape)

# Important:
# Do not automatically delete statistical outliers.
# Large count values can be valid website observations.
# Inspect, transform, or use robust/tree methods instead.

## 6. Domain-grouped train / validation / test split

In [ ]:
# Make this cell self-contained: create Domain_normalized if it does not exist.
def _normalize_domain_for_split(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip().lower()
    if not text:
        return np.nan

    parse_text = text if "://" in text else "http://" + text

    try:
        host = urlsplit(parse_text).hostname
        if host:
            return host.lower().rstrip(".")
    except Exception:
        pass

    text = re.sub(r"^https?://", "", text)
    text = text.split("/")[0]
    text = text.split(":")[0]
    return text.rstrip(".") or np.nan


# Ensure the normalized grouping column exists.
if "Domain_normalized" not in df_clean.columns:
    print("Domain_normalized not found. Creating it now...")

    if "Domain" in df_clean.columns:
        df_clean["Domain_normalized"] = df_clean["Domain"].map(
            _normalize_domain_for_split
        )
    elif "URL" in df_clean.columns:
        df_clean["Domain_normalized"] = df_clean["URL"].map(
            _normalize_domain_for_split
        )
    else:
        raise KeyError(
            "Neither 'Domain' nor 'URL' exists, so domain groups cannot be created."
        )

# Replace remaining missing domains with unique synthetic groups.
missing_mask = df_clean["Domain_normalized"].isna()

if missing_mask.any():
    df_clean.loc[missing_mask, "Domain_normalized"] = [
        f"__missing_domain_{i}"
        for i in df_clean.index[missing_mask]
    ]

print("Unique domain groups:", df_clean["Domain_normalized"].nunique())

groups = df_clean["Domain_normalized"]

# Approximate 70/15/15 domain-grouped split.
gss1 = GroupShuffleSplit(
    n_splits=1,
    train_size=0.70,
    random_state=RANDOM_STATE
)

train_idx, temp_idx = next(
    gss1.split(
        df_clean,
        y=df_clean[TARGET],
        groups=groups
    )
)

train_df = df_clean.iloc[train_idx].copy()
temp_df = df_clean.iloc[temp_idx].copy()

gss2 = GroupShuffleSplit(
    n_splits=1,
    train_size=0.50,
    random_state=RANDOM_STATE
)

val_rel_idx, test_rel_idx = next(
    gss2.split(
        temp_df,
        y=temp_df[TARGET],
        groups=temp_df["Domain_normalized"]
    )
)

val_df = temp_df.iloc[val_rel_idx].copy()
test_df = temp_df.iloc[test_rel_idx].copy()


def split_summary(name, frame):
    return {
        "split": name,
        "rows": len(frame),
        "row_pct": 100 * len(frame) / len(df_clean),
        "phishing_count": int(frame[TARGET].sum()),
        "phishing_rate": float(frame[TARGET].mean()),
        "unique_domains": frame["Domain_normalized"].nunique(),
    }


split_table = pd.DataFrame([
    split_summary("train", train_df),
    split_summary("validation", val_df),
    split_summary("test", test_df),
])

display(split_table.round(4))

train_groups = set(train_df["Domain_normalized"])
val_groups = set(val_df["Domain_normalized"])
test_groups = set(test_df["Domain_normalized"])

assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)

print("Confirmed: no normalized domain overlap across splits.")

> PhiUSIIL does not provide an observation-date variable in the published feature inventory, so this notebook does not claim a chronological split. A grouped-domain split is used to reduce optimistic leakage from related domains.

## 7. Applied EDA

In [ ]:
counts = df_clean[TARGET].value_counts().sort_index()
labels = ["Legitimate (0)", "Phishing (1)"]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(labels, [counts.get(0, 0), counts.get(1, 0)])
ax.set_title("Class Distribution")
ax.set_ylabel("Number of observations")

for i, value in enumerate([counts.get(0, 0), counts.get(1, 0)]):
    ax.text(i, value, f"{value:,}", ha="center", va="bottom")

plt.tight_layout()
plt.show()

print("Phishing prevalence:", round(df_clean[TARGET].mean(), 4))

In [ ]:
if "URLLength" in df_clean.columns:
    fig, ax = plt.subplots(figsize=(9, 5))
    display_max = df_clean["URLLength"].quantile(0.99)

    for target_value, label_name in [(0, "Legitimate"), (1, "Phishing")]:
        x = df_clean.loc[
            df_clean[TARGET] == target_value, "URLLength"
        ].dropna()
        ax.hist(
            x.clip(upper=display_max),
            bins=50,
            alpha=0.55,
            label=label_name
        )

    ax.set_title(
        "URL Length Distribution by Class "
        "(display clipped at 99th percentile)"
    )
    ax.set_xlabel("URL length")
    ax.set_ylabel("Frequency")
    ax.legend()
    plt.tight_layout()
    plt.show()

    display(
        df_clean.groupby(TARGET)["URLLength"]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .rename(index={0: "Legitimate", 1: "Phishing"})
    )

In [ ]:
if "TLD" in df_clean.columns:
    tld_summary = (
        df_clean.groupby("TLD")[TARGET]
        .agg(["count", "mean"])
        .rename(columns={"mean": "phishing_rate"})
        .sort_values("count", ascending=False)
    )

    top_tlds = tld_summary.head(20).sort_values("phishing_rate")

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(top_tlds.index.astype(str), top_tlds["phishing_rate"])
    ax.set_title("Phishing Rate among the 20 Most Frequent TLDs")
    ax.set_xlabel("Phishing rate")
    plt.tight_layout()
    plt.show()

    display(tld_summary.head(25))

In [ ]:
for feature in ["IsDomainIP", "IsHTTPS", "HasObfuscation"]:
    if feature in df_clean.columns:
        summary = (
            df_clean.groupby(feature)[TARGET]
            .agg(["count", "mean"])
            .rename(columns={"mean": "phishing_rate"})
        )
        print(f"\n{feature}")
        display(summary)

In [ ]:
web_count_features = [
    "LineOfCode", "LargestLineLength",
    "NoOfURLRedirect", "NoOfSelfRedirect",
    "NoOfPopup", "NoOfiFrame", "NoOfImage",
    "NoOfCSS", "NoOfJS", "NoOfSelfRef",
    "NoOfEmptyRef", "NoOfExternalRef"
]
web_count_features = [c for c in web_count_features if c in df_clean.columns]

if web_count_features:
    summary = df_clean[web_count_features].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    ).T
    display(summary)

    for feature in web_count_features[:6]:
        fig, ax = plt.subplots(figsize=(8, 4))

        for target_value, label_name in [(0, "Legitimate"), (1, "Phishing")]:
            values = np.log1p(
                df_clean.loc[
                    df_clean[TARGET] == target_value, feature
                ].clip(lower=0).dropna()
            )
            ax.hist(values, bins=50, alpha=0.55, label=label_name)

        ax.set_title(f"log1p({feature}) by class")
        ax.set_xlabel("log1p value")
        ax.set_ylabel("Frequency")
        ax.legend()
        plt.tight_layout()
        plt.show()

In [ ]:
numeric = df_clean.select_dtypes(include=np.number).copy()
numeric = numeric.drop(columns=["label"], errors="ignore")

corr_to_target = (
    numeric.corr()[TARGET]
    .drop(TARGET)
    .abs()
    .sort_values(ascending=False)
)

top_corr_features = corr_to_target.head(20).index.tolist()
corr_matrix = numeric[top_corr_features + [TARGET]].corr()

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr_matrix.values, aspect="auto")
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr_matrix.index)))
ax.set_yticklabels(corr_matrix.index, fontsize=8)
ax.set_title("Correlation Heatmap — Top Numeric Features")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

display(
    numeric.corr()[TARGET]
    .drop(TARGET)
    .sort_values(key=np.abs, ascending=False)
    .head(20)
    .to_frame("correlation_with_is_phishing")
)

## 8. Safe URL-only feature engineering

In [ ]:
SUSPICIOUS_TOKENS = [
    "login", "verify", "verification", "secure", "account", "update",
    "signin", "bank", "confirm", "password", "wallet", "crypto",
    "invoice", "payment", "support", "recover", "unlock"
]

def shannon_entropy(text):
    text = str(text)
    if not text:
        return 0.0

    values = pd.Series(list(text)).value_counts(normalize=True)
    return float(-(values * np.log2(values)).sum())


def is_ip_host(host):
    if not host:
        return 0

    try:
        ipaddress.ip_address(host)
        return 1
    except ValueError:
        return 0


def safe_url_features(url):
    raw = "" if pd.isna(url) else str(url).strip()
    parse_text = raw if "://" in raw else "http://" + raw

    try:
        p = urlsplit(parse_text)
        host = p.hostname or ""
        path = p.path or ""
        query = p.query or ""
        scheme = (p.scheme or "").lower()
    except Exception:
        host, path, query, scheme = "", "", "", ""

    letters = sum(ch.isalpha() for ch in raw)
    digits = sum(ch.isdigit() for ch in raw)

    punctuation_chars = ".-_~:/?#[]@!$&'()*+,;=%"
    punctuation_count = sum(ch in punctuation_chars for ch in raw)

    token_hits = sum(
        token in raw.lower()
        for token in SUSPICIOUS_TOKENS
    )

    return {
        "eng_url_char_len": len(raw),
        "eng_hostname_len": len(host),
        "eng_path_len": len(path),
        "eng_query_len": len(query),
        "eng_dot_count": raw.count("."),
        "eng_hyphen_count": raw.count("-"),
        "eng_at_count": raw.count("@"),
        "eng_slash_count": raw.count("/"),
        "eng_question_count": raw.count("?"),
        "eng_equal_count": raw.count("="),
        "eng_ampersand_count": raw.count("&"),
        "eng_percent_count": raw.count("%"),
        "eng_digit_count": digits,
        "eng_letter_count": letters,
        "eng_digit_letter_ratio": digits / max(letters, 1),
        "eng_punctuation_count": punctuation_count,
        "eng_punctuation_ratio": punctuation_count / max(len(raw), 1),
        "eng_path_depth": len(
            [part for part in path.split("/") if part]
        ),
        "eng_query_param_count": len(
            parse_qsl(query, keep_blank_values=True)
        ),
        "eng_entropy": shannon_entropy(raw),
        "eng_suspicious_token_count": token_hits,
        "eng_uses_https": int(scheme == "https"),
        "eng_host_is_ip": is_ip_host(host),
    }


if "URL" not in df_clean.columns:
    raise KeyError("URL column is required for lexical feature engineering.")

url_eng = (
    df_clean["URL"]
    .apply(safe_url_features)
    .apply(pd.Series)
)

df_fe = pd.concat(
    [
        df_clean.reset_index(drop=True),
        url_eng.reset_index(drop=True)
    ],
    axis=1
)

print("Added engineered URL-only features:", url_eng.shape[1])
display(df_fe.filter(regex=r"^eng_").head())

## 9. Full-feature engineering

In [ ]:
punct_cols = [
    "NoOfEqualsInURL",
    "NoOfQMarkInURL",
    "NoOfAmpersandInURL",
    "NoOfOtherSpecialCharsInURL"
]

if all(c in df_fe.columns for c in punct_cols):
    df_fe["eng_dataset_punctuation_total"] = df_fe[punct_cols].sum(axis=1)

if "URLLength" in df_fe.columns:
    df_fe["eng_url_length_band"] = pd.cut(
        df_fe["URLLength"],
        bins=[-np.inf, 50, 100, 150, 250, np.inf],
        labels=[
            "very_short", "short", "medium",
            "long", "very_long"
        ]
    )

if "NoOfSubDomain" in df_fe.columns:
    df_fe["eng_subdomain_depth_band"] = pd.cut(
        df_fe["NoOfSubDomain"],
        bins=[-np.inf, 0, 1, 2, 3, np.inf],
        labels=["none", "one", "two", "three", "four_plus"]
    )

if {"NoOfExternalRef", "NoOfSelfRef"}.issubset(df_fe.columns):
    df_fe["eng_external_self_ref_ratio"] = (
        df_fe["NoOfExternalRef"] /
        (df_fe["NoOfSelfRef"] + 1)
    )

complexity_cols = [
    c for c in ["NoOfImage", "NoOfCSS", "NoOfJS", "NoOfiFrame"]
    if c in df_fe.columns
]

if complexity_cols:
    df_fe["eng_webpage_complexity_index"] = (
        df_fe[complexity_cols].sum(axis=1)
    )

form_risk_cols = [
    c for c in [
        "HasPasswordField",
        "HasHiddenFields",
        "HasExternalFormSubmit",
        "HasSubmitButton"
    ]
    if c in df_fe.columns
]

if form_risk_cols:
    df_fe["eng_form_risk_index"] = (
        df_fe[form_risk_cols].sum(axis=1)
    )

commerce_cols = [
    c for c in ["Bank", "Pay", "Crypto"]
    if c in df_fe.columns
]

if commerce_cols:
    df_fe["eng_commerce_indicator"] = (
        df_fe[commerce_cols].sum(axis=1) > 0
    ).astype(int)

for c in web_count_features:
    if c in df_fe.columns:
        df_fe[f"eng_log1p_{c}"] = np.log1p(
            df_fe[c].clip(lower=0)
        )

print("Feature-engineered shape:", df_fe.shape)
display(df_fe.filter(regex=r"^eng_").head())

## 10. Define URL-only and full-feature experiment sets

In [ ]:
published_url_only = [
    "URLLength", "DomainLength", "IsDomainIP", "TLDLength",
    "NoOfSubDomain", "NoOfLettersInURL", "LetterRatioInURL",
    "NoOfDegitsInURL", "DegitRatioInURL", "NoOfEqualsInURL",
    "NoOfQMarkInURL", "NoOfAmpersandInURL",
    "NoOfOtherSpecialCharsInURL", "SpacialCharRatioInURL",
    "IsHTTPS", "HasObfuscation", "NoOfObfuscatedChar",
    "ObfuscationRatio"
]

derived_similarity_probability = [
    "URLSimilarityIndex",
    "CharContinuationRate",
    "TLDLegitimateProb",
    "URLCharProb"
]

full_only_markers = [
    "webpage_complexity",
    "form_risk",
    "external_self_ref",
    "commerce",
    "log1p_LineOfCode",
    "log1p_LargestLineLength",
    "log1p_NoOfImage",
    "log1p_NoOfCSS",
    "log1p_NoOfJS",
    "log1p_NoOfSelfRef",
    "log1p_NoOfEmptyRef",
    "log1p_NoOfExternalRef",
    "log1p_NoOfURLRedirect",
    "log1p_NoOfSelfRedirect",
    "log1p_NoOfPopup",
    "log1p_NoOfiFrame"
]

engineered_url_only = [
    c for c in df_fe.columns
    if c.startswith("eng_")
    and not any(marker in c for marker in full_only_markers)
]

url_only_features = [
    c for c in published_url_only + engineered_url_only
    if c in df_fe.columns
    and pd.api.types.is_numeric_dtype(df_fe[c])
]

non_model = {
    "FILENAME", "URL", "Domain", "TLD", "Title",
    "URL_normalized", "Domain_normalized",
    "label", TARGET
}

full_numeric_features = [
    c for c in df_fe.select_dtypes(include=np.number).columns
    if c not in non_model
]

print("URL-only numeric features:", len(url_only_features))
print("Full-feature numeric features:", len(full_numeric_features))
print("\nSimilarity/probability variables reserved for ablation:")
print([
    c for c in derived_similarity_probability
    if c in df_fe.columns
])

## 11. Re-create split membership after feature engineering

In [ ]:
train_rows = set(train_df.index)
val_rows = set(val_df.index)
test_rows = set(test_df.index)

fe_train = df_fe.loc[df_fe.index.isin(train_rows)].copy()
fe_val = df_fe.loc[df_fe.index.isin(val_rows)].copy()
fe_test = df_fe.loc[df_fe.index.isin(test_rows)].copy()

print(
    "Rows:",
    len(fe_train),
    len(fe_val),
    len(fe_test)
)

print(
    "Phishing rates:",
    round(fe_train[TARGET].mean(), 4),
    round(fe_val[TARGET].mean(), 4),
    round(fe_test[TARGET].mean(), 4)
)

## 12. Feature selection

In [ ]:
def correlation_prune(frame, features, threshold=0.95):
    x = frame[features].copy()
    x = x.select_dtypes(include=np.number)

    corr = x.corr().abs()
    upper = corr.where(
        np.triu(np.ones(corr.shape), k=1).astype(bool)
    )

    to_drop = [
        column for column in upper.columns
        if any(upper[column] > threshold)
    ]

    keep = [
        c for c in x.columns
        if c not in to_drop
    ]

    return keep, to_drop


url_keep_corr, url_drop_corr = correlation_prune(
    fe_train,
    url_only_features,
    threshold=0.95
)

print(
    "URL-only features before correlation pruning:",
    len(url_only_features)
)
print(
    "URL-only features retained:",
    len(url_keep_corr)
)
print(
    "Highly correlated features flagged:",
    len(url_drop_corr)
)

display(
    pd.DataFrame({
        "flagged_for_review": url_drop_corr
    }).head(30)
)

In [ ]:
X_train_mi = (
    fe_train[url_keep_corr]
    .replace([np.inf, -np.inf], np.nan)
)
y_train = fe_train[TARGET].astype(int)

mi_imputer = SimpleImputer(strategy="median")
X_train_mi_imp = mi_imputer.fit_transform(X_train_mi)

mi_scores = mutual_info_classif(
    X_train_mi_imp,
    y_train,
    random_state=RANDOM_STATE
)

mi_table = (
    pd.DataFrame({
        "feature": url_keep_corr,
        "mutual_information": mi_scores
    })
    .sort_values("mutual_information", ascending=False)
    .reset_index(drop=True)
)

display(mi_table.head(25))

fig, ax = plt.subplots(figsize=(9, 7))
plot_mi = (
    mi_table.head(20)
    .sort_values("mutual_information")
)

ax.barh(
    plot_mi["feature"],
    plot_mi["mutual_information"]
)
ax.set_title("Top 20 URL-only Features by Mutual Information")
ax.set_xlabel("Mutual information")
plt.tight_layout()
plt.show()

In [ ]:
X_train_l1 = (
    fe_train[url_keep_corr]
    .replace([np.inf, -np.inf], np.nan)
)

l1_imputer = SimpleImputer(strategy="median")
l1_scaler = StandardScaler()

X_l1_imp = l1_imputer.fit_transform(X_train_l1)
X_l1_scaled = l1_scaler.fit_transform(X_l1_imp)

l1_model = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    C=0.1,
    class_weight="balanced",
    max_iter=2000,
    random_state=RANDOM_STATE
)

l1_model.fit(X_l1_scaled, y_train)

l1_table = (
    pd.DataFrame({
        "feature": url_keep_corr,
        "coefficient": l1_model.coef_[0],
        "abs_coefficient": np.abs(l1_model.coef_[0])
    })
    .sort_values("abs_coefficient", ascending=False)
    .reset_index(drop=True)
)

selected_l1 = (
    l1_table.loc[
        l1_table["abs_coefficient"] > 0,
        "feature"
    ]
    .tolist()
)

print(
    "Features with non-zero L1 coefficients:",
    len(selected_l1)
)
display(l1_table.head(25))

## 13. Dimensionality reduction — PCA

In [ ]:
pca_features = [
    c for c in full_numeric_features
    if c in fe_train.columns
]

X_pca_train = (
    fe_train[pca_features]
    .replace([np.inf, -np.inf], np.nan)
)

pca_imputer = SimpleImputer(strategy="median")
pca_scaler = StandardScaler()

X_pca_imp = pca_imputer.fit_transform(X_pca_train)
X_pca_scaled = pca_scaler.fit_transform(X_pca_imp)

pca = PCA()
pca.fit(X_pca_scaled)

cum_var = np.cumsum(
    pca.explained_variance_ratio_
)

n95 = int(
    np.argmax(cum_var >= 0.95) + 1
)

print(
    f"Components required for >=95% explained variance: {n95}"
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    np.arange(1, len(cum_var) + 1),
    cum_var
)
ax.axhline(0.95, linestyle="--")
ax.axvline(n95, linestyle="--")
ax.set_title("PCA Cumulative Explained Variance")
ax.set_xlabel("Number of components")
ax.set_ylabel("Cumulative explained variance")
plt.tight_layout()
plt.show()

In [ ]:
loading_matrix = pd.DataFrame(
    pca.components_.T,
    index=pca_features,
    columns=[
        f"PC{i+1}"
        for i in range(len(pca.components_))
    ]
)

for pc in ["PC1", "PC2", "PC3"]:
    if pc in loading_matrix.columns:
        print(f"\nTop absolute loadings for {pc}:")
        display(
            loading_matrix[pc]
            .sort_values(
                key=np.abs,
                ascending=False
            )
            .head(10)
            .to_frame()
        )

## 14. Optional t-SNE visualization

In [ ]:
sample_n = min(5000, len(fe_train))
sample = fe_train.sample(
    sample_n,
    random_state=RANDOM_STATE
)

X_tsne = (
    sample[pca_features]
    .replace([np.inf, -np.inf], np.nan)
)

X_tsne = pca_imputer.transform(X_tsne)
X_tsne = pca_scaler.transform(X_tsne)

pca_30 = PCA(
    n_components=min(30, X_tsne.shape[1]),
    random_state=RANDOM_STATE
)
X_tsne_pca = pca_30.fit_transform(X_tsne)

tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=RANDOM_STATE
)

embedding = tsne.fit_transform(X_tsne_pca)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    embedding[:, 0],
    embedding[:, 1],
    c=sample[TARGET].values,
    s=8,
    alpha=0.6
)

ax.set_title("t-SNE Visualization of Training Sample")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")

legend = ax.legend(
    *scatter.legend_elements(),
    title="is_phishing"
)
ax.add_artist(legend)

plt.tight_layout()
plt.show()

print(
    "Do not interpret visible separation as proof of deployment accuracy. "
    "Predictive performance must be measured on held-out groups."
)

## 15. Diagnostic model-based feature importance

In [ ]:
# Diagnostic only — not the final Step 4 champion model.

rf_features = url_keep_corr

X_rf = (
    fe_train[rf_features]
    .replace([np.inf, -np.inf], np.nan)
)

rf_imputer = SimpleImputer(strategy="median")
X_rf_imp = rf_imputer.fit_transform(X_rf)

rf_diag = RandomForestClassifier(
    n_estimators=250,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

rf_diag.fit(
    X_rf_imp,
    y_train
)

importance_table = (
    pd.DataFrame({
        "feature": rf_features,
        "rf_importance": rf_diag.feature_importances_
    })
    .sort_values("rf_importance", ascending=False)
    .reset_index(drop=True)
)

display(importance_table.head(25))

fig, ax = plt.subplots(figsize=(9, 7))
plot_imp = (
    importance_table.head(20)
    .sort_values("rf_importance")
)

ax.barh(
    plot_imp["feature"],
    plot_imp["rf_importance"]
)
ax.set_title(
    "Diagnostic Random Forest — Top URL-only Feature Importances"
)
ax.set_xlabel("Feature importance")
plt.tight_layout()
plt.show()

## 16. Optional SHAP diagnostic

In [ ]:
# Optional dependency:
# %pip install shap

try:
    import shap

    sample_n = min(1000, len(fe_train))
    shap_sample = fe_train[rf_features].sample(
        sample_n,
        random_state=RANDOM_STATE
    )

    shap_sample_imp = rf_imputer.transform(
        shap_sample.replace(
            [np.inf, -np.inf],
            np.nan
        )
    )

    explainer = shap.TreeExplainer(rf_diag)
    shap_values = explainer.shap_values(
        shap_sample_imp
    )

    values = shap_values

    if isinstance(shap_values, list):
        values = shap_values[1]
    elif (
        hasattr(shap_values, "ndim")
        and shap_values.ndim == 3
    ):
        values = shap_values[:, :, 1]

    shap.summary_plot(
        values,
        shap_sample_imp,
        feature_names=rf_features,
        show=True
    )

except ImportError:
    print(
        "SHAP is not installed. "
        "Run '%pip install shap' if required."
    )
except Exception as e:
    print(
        "SHAP diagnostic skipped due to:",
        repr(e)
    )

## 17. Export reproducible Step 3 outputs

In [ ]:
split_map = pd.Series(
    index=df_fe.index,
    dtype="object"
)

split_map.loc[fe_train.index] = "train"
split_map.loc[fe_val.index] = "validation"
split_map.loc[fe_test.index] = "test"

df_fe["split"] = split_map

df_fe.to_csv(
    OUTPUT_DIR / "phiusiil_step3_processed.csv",
    index=False
)

try:
    df_fe.to_parquet(
        OUTPUT_DIR / "phiusiil_step3_processed.parquet",
        index=False
    )
    print("Saved Parquet output.")
except Exception:
    print(
        "Parquet not saved "
        "(pyarrow/fastparquet may not be installed)."
    )

quality_missing.to_csv(
    OUTPUT_DIR / "data_quality_missing_audit.csv"
)
mi_table.to_csv(
    OUTPUT_DIR / "mutual_information_scores.csv",
    index=False
)
l1_table.to_csv(
    OUTPUT_DIR / "l1_feature_selection.csv",
    index=False
)
importance_table.to_csv(
    OUTPUT_DIR / "rf_diagnostic_feature_importance.csv",
    index=False
)
split_table.to_csv(
    OUTPUT_DIR / "split_summary.csv",
    index=False
)

metadata = {
    "dataset_file": DATA_PATH.name,
    "dataset_sha256": sha256sum(DATA_PATH),
    "raw_rows": int(df_raw.shape[0]),
    "raw_columns": int(df_raw.shape[1]),
    "processed_rows": int(df_fe.shape[0]),
    "processed_columns": int(df_fe.shape[1]),
    "target": TARGET,
    "positive_class": "phishing",
    "random_state": RANDOM_STATE,
    "split_strategy": "domain-grouped approximate 70/15/15",
    "url_only_feature_count": len(url_only_features),
    "full_numeric_feature_count": len(full_numeric_features),
    "correlation_pruned_url_feature_count": len(url_keep_corr),
    "l1_selected_feature_count": len(selected_l1),
    "pca_components_95pct_variance": n95
}

with open(
    OUTPUT_DIR / "step3_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, indent=2)

print(
    f"Outputs saved to: {OUTPUT_DIR.resolve()}"
)
display(
    pd.Series(
        metadata,
        name="value"
    ).to_frame()
)

## 18. Report-ready findings checklist

After running the notebook, capture these items in the **EDA + Feature Engineering Report**:

- Final row/column count after exact-duplicate removal.
- Missing, blank, infinite, and invalid-value findings.
- Duplicate URL/domain counts and conflicting labels.
- Train/validation/test sizes and phishing prevalence.
- Confirmation that domains do not overlap across splits.
- URL-length, TLD, IP-domain, HTTPS, obfuscation, and webpage-complexity findings.
- Outlier treatment decisions.
- Engineered features and their cybersecurity rationale.
- Correlation-pruning, mutual-information, and L1-selection results.
- PCA explained variance and principal-component loadings.
- t-SNE visualization with the caveat that visual separation does not prove predictive accuracy.
- Diagnostic feature importance / SHAP results.
- Leakage concerns related to similarity/probability and webpage-derived features.
- Clear distinction between:
  - **Experiment A:** URL-only, safe pre-click features.
  - **Experiment B:** full-feature analytical upper-bound model.

The locked test set should be used for final performance reporting only after model selection and calibration in **Step 4 — Model Implementation**.